In [1]:
!pip install tensorflow

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os

In [4]:
!pip install bing-image-downloader

In [5]:
from bing_image_downloader import downloader

downloader.download("damaged road potholes", limit=40, output_dir="dataset")
downloader.download("road asphalt cracks", limit=40, output_dir="dataset")
downloader.download("smooth asphalt road", limit=40, output_dir="dataset")

[%] Downloading Images to /content/dataset/damaged road potholes


[!!]Indexing page: 1

[%] Indexed 35 Images on Page 1.


[%] Downloading Image #1 from https://as2.ftcdn.net/v2/jpg/06/84/96/71/1000_F_684967130_EYfhfGJLXDjdZO644HJawGbecaFvVxbZ.jpg
[%] File Downloaded !

[%] Downloading Image #2 from https://c8.alamy.com/comp/CC4PRG/road-damaged-by-potholes-CC4PRG.jpg
[%] File Downloaded !

[%] Downloading Image #3 from https://c8.alamy.com/comp/CC4PT6/road-damaged-by-potholes-CC4PT6.jpg
[%] File Downloaded !

[%] Downloading Image #4 from https://c8.alamy.com/comp/CC4PT1/road-damaged-by-potholes-CC4PT1.jpg
[%] File Downloaded !

[%] Downloading Image #5 from https://as1.ftcdn.net/v2/jpg/09/80/32/60/1000_F_980326097_GJnwKBxRy3zJZ2BvBQIm8USLlrsdwdgi.jpg
[%] File Downloaded !

[%] Downloading Image #6 from https://image.shutterstock.com/shutterstock/photos/1134687086/display_1500/stock-photo-damaged-asphalt-pavement-road-with-potholes-1134687086.jpg
[%] File Downloaded !

[%] Downloading

In [6]:
import os

os.rename("dataset/damaged road potholes","dataset/damage")
os.rename("dataset/road asphalt cracks","dataset/moderate")
os.rename("dataset/smooth asphalt road","dataset/smooth")

In [7]:
print(os.listdir("dataset"))

['moderate', 'damage', 'smooth']


In [8]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset",
    image_size=(128,128),
    batch_size=8,
    validation_split=0.2,
    subset="training",
    seed=123
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset",
    image_size=(128,128),
    batch_size=8,
    validation_split=0.2,
    subset="validation",
    seed=123
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 120 files belonging to 3 classes.
Using 96 files for training.
Found 120 files belonging to 3 classes.
Using 24 files for validation.
Classes: ['damage', 'moderate', 'smooth']


In [9]:
from tensorflow.keras import layers

normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x,y:(normalization_layer(x),y))
val_ds = val_ds.map(lambda x,y:(normalization_layer(x),y))

In [10]:
from tensorflow.keras import models

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128,128,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [11]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')
])

In [12]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 446ms/step - accuracy: 0.4477 - loss: 1.3662 - val_accuracy: 0.6250 - val_loss: 0.7798
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 276ms/step - accuracy: 0.8529 - loss: 0.4210 - val_accuracy: 0.7917 - val_loss: 0.5013
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 410ms/step - accuracy: 0.9467 - loss: 0.1805 - val_accuracy: 0.7917 - val_loss: 0.4745
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 277ms/step - accuracy: 0.9872 - loss: 0.1274 - val_accuracy: 0.7500 - val_loss: 0.5001
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 277ms/step - accuracy: 0.9975 - loss: 0.0672 - val_accuracy: 0.7917 - val_loss: 0.4615
Epoch 6/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 361ms/step - accuracy: 1.0000 - loss: 0.0308 - val_accuracy: 0.7917 - val_loss: 0.4557
Epoch 7/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 291ms/step - accuracy: 1.0000 - loss: 0.0381 - val_accuracy: 0.7500 - val_loss: 0.4747
Epoch 8/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 275ms/step - accuracy: 1.0000 - loss: 0.0173 - val_accuracy: 0

In [15]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))

In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/RoadSurfaceProject/models", exist_ok=True)

model.save("/content/drive/MyDrive/RoadSurfaceProject/models/road_model.keras")

print("Model saved successfully!")

Mounted at /content/drive
Model saved successfully!
